# Quartz Solubility Analysis using DEW2024

This tutorial demonstrates how to calculate quartz solubility at high temperatures and pressures using Reaktoro's DEW (Deep Earth Water) model and compare results with experimental data.

**What you'll learn:**
- Load thermodynamic databases (DEW2024 and SUPCRTBL)
- Build a chemical system with aqueous and mineral phases
- Calculate quartz solubility across temperature-pressure conditions
- Compare model predictions with experimental data
- Create publication-quality plots

## Step 1: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from reaktoro import *
import os

# Suppress warnings for cleaner output
try:
    Warnings.disable(906)
except:
    pass

print("✓ Libraries imported successfully")

## Step 2: Configuration and Helper Functions

Define temperature/pressure ranges and saturation pressure functions.

In [ ]:
# Configuration
T_MIN, T_MAX = 150, 550
N_POINTS = 100

def psat_bar(T_C):
    """Calculate water saturation pressure (bar) using Antoine equation."""
    if T_C < 0 or T_C > 374:
        return np.nan
    T_K = T_C + 273.15
    A, B, C = 5.40221, 1838.675, -31.737
    log10_P = A - B / (T_K + C)
    return 10**log10_P

def psat_kbar(T_C):
    """Calculate water saturation pressure (kbar)."""
    P = psat_bar(T_C)
    return P / 1000.0 if not np.isnan(P) else np.nan

print("✓ Configuration loaded")

## Step 3: Load Experimental Data

Load experimental quartz solubility measurements from various studies.

In [ ]:
# Path to experimental data
CSV_FILE = "DEW_Experimental_Benchmark/quartz_DEW_testset.csv"

if os.path.exists(CSV_FILE):
    exp_data = pd.read_csv(CSV_FILE)
    exp_data = exp_data[["T_C", "P_kbar", "molality_m", "reference", "experiment_type"]].copy()
    exp_data["P_bar"] = exp_data["P_kbar"] * 1000.0

    # Mark experiments on saturation pressure curve
    kennedy_controlled_pressures = {0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.6, 0.75}
    is_kennedy = exp_data["reference"].str.contains("Kennedy", case=False, na=False)
    is_kennedy_controlled = (
        is_kennedy &
        exp_data["P_kbar"].notna() &
        exp_data["P_kbar"].isin(kennedy_controlled_pressures)
    )
    exp_data["is_psat"] = ~is_kennedy_controlled

    print(f"✓ Loaded {len(exp_data)} experimental data points")
    print(f"  Temperature range: {exp_data['T_C'].min():.0f} - {exp_data['T_C'].max():.0f} °C")
    print(f"  Pressure range: {exp_data['P_kbar'].min():.3f} - {exp_data['P_kbar'].max():.3f} kbar")
else:
    print(f"⚠ Experimental data file not found: {CSV_FILE}")
    exp_data = pd.DataFrame()

## Step 4: Build Chemical System

Load databases and create the chemical system with DEW aqueous phase and quartz mineral.

In [ ]:
print("Building chemical system...")

# Load databases
dew_db = DEWDatabase("dew2024-aqueous")
supcrt_db = SupcrtDatabase("supcrtbl")

# Combine databases
quartz_species = supcrt_db.species("Quartz")
combined_db = Database(dew_db.species())
combined_db.addSpecies(quartz_species)

# Define phases
aqueous = AqueousPhase("WATER,AQ H+ OH- SiO2_aq H2_aq O2_aq HO2- HSiO3- Si2O4_aq Si3O6_aq")
aqueous.setActivityModel(ActivityModelDEW())
mineral = MineralPhase("Quartz")

# Create system
system = ChemicalSystem(combined_db, aqueous, mineral)

print(f"✓ Chemical system built: {system.species().size()} species")
print("✓ DEW activity model configured")

## Step 5: Calculate Solubility Curves

Calculate quartz solubility at each experimental pressure using the modern EquilibriumSpecs pattern.

In [ ]:
print("\nCalculating solubility curves...")
print("=" * 70)

solubility_curves = {}

if len(exp_data) > 0:
    # Get unique pressures from experimental data
    pressures_kbar = sorted(exp_data["P_kbar"].dropna().unique())

    for P_kbar in pressures_kbar:
        P_bar = P_kbar * 1000.0
        print(f"P = {P_kbar:.3f} kbar ({P_bar:.0f} bar)...")

        # Determine temperature range from experiments at this pressure
        P_tol = 0.05 * P_kbar
        exp_at_P = exp_data[
            (exp_data["P_kbar"] >= P_kbar - P_tol) &
            (exp_data["P_kbar"] <= P_kbar + P_tol)
        ]

        if len(exp_at_P) > 0:
            T_min_cat = exp_at_P["T_C"].min()
            T_max_cat = exp_at_P["T_C"].max()
            T_span = T_max_cat - T_min_cat
            T_min = max(25, T_min_cat - 0.05 * T_span) if T_span > 0 else T_min_cat - 50
            T_max = min(1000, T_max_cat + 0.05 * T_span) if T_span > 0 else T_max_cat + 50
        else:
            T_min, T_max = T_MIN, T_MAX

        T_range = np.linspace(T_min, T_max, N_POINTS)

        # Setup equilibrium solver
        specs = EquilibriumSpecs(system)
        specs.temperature()
        specs.pressure()
        solver = EquilibriumSolver(specs)
        conditions = EquilibriumConditions(specs)

        # Initial state
        state = ChemicalState(system)
        state.set("WATER,AQ", 1.0, "kg")
        state.set("H+", 1e-8, "mol")
        state.set("OH-", 1e-8, "mol")
        state.set("SiO2_aq", 1e-6, "mol")
        state.set("Quartz", 10.0, "mol")

        # Calculate solubility at each temperature
        molalities = []
        for T_C in T_range:
            conditions.temperature(float(T_C), "celsius")
            conditions.pressure(float(P_bar), "bar")
            result = solver.solve(state, conditions)

            if result.succeeded():
                try:
                    aqprops = AqueousProps(state)
                    molality = float(aqprops.speciesMolality("SiO2_aq"))
                except:
                    molality = np.nan
            else:
                molality = np.nan

            molalities.append(molality)

        solubility_curves[P_kbar] = {
            "T_C": T_range,
            "molality": np.array(molalities)
        }

        valid_points = np.sum(~np.isnan(molalities))
        print(f"  ✓ Calculated {valid_points}/{N_POINTS} points")

print("=" * 70)

## Step 6: Calculate Psat Curve

Calculate solubility along the water saturation pressure curve.

In [ ]:
print("\nCalculating Psat curve...")

if len(exp_data) > 0:
    psat_data = exp_data[exp_data["is_psat"]]
    if len(psat_data) > 0:
        T_psat_min = max(25, psat_data["T_C"].min() - 25)
        T_psat_max = min(374, psat_data["T_C"].max() + 25)
    else:
        T_psat_min, T_psat_max = 100, 374
else:
    T_psat_min, T_psat_max = 100, 374

T_psat_range = np.linspace(T_psat_min, T_psat_max, N_POINTS)
P_psat_values = np.array([psat_kbar(T) for T in T_psat_range])
valid_temps = ~np.isnan(P_psat_values)

# Setup equilibrium solver for Psat
specs_psat = EquilibriumSpecs(system)
specs_psat.temperature()
specs_psat.pressure()
solver_psat = EquilibriumSolver(specs_psat)
conditions_psat = EquilibriumConditions(specs_psat)

state_psat = ChemicalState(system)
state_psat.set("WATER,AQ", 1.0, "kg")
state_psat.set("H+", 1e-8, "mol")
state_psat.set("OH-", 1e-8, "mol")
state_psat.set("SiO2_aq", 1e-6, "mol")
state_psat.set("Quartz", 10.0, "mol")

psat_molalities = []
for i, T_C in enumerate(T_psat_range):
    if not valid_temps[i]:
        psat_molalities.append(np.nan)
        continue

    P_bar = P_psat_values[i] * 1000.0
    conditions_psat.temperature(float(T_C), "celsius")
    conditions_psat.pressure(float(P_bar), "bar")
    result = solver_psat.solve(state_psat, conditions_psat)

    if result.succeeded():
        try:
            aqprops = AqueousProps(state_psat)
            molality = float(aqprops.speciesMolality("SiO2_aq"))
        except:
            molality = np.nan
    else:
        molality = np.nan

    psat_molalities.append(molality)

solubility_curves["Psat"] = {
    "T_C": T_psat_range,
    "P_kbar": P_psat_values,
    "molality": np.array(psat_molalities)
}

valid_psat = np.sum(~np.isnan(psat_molalities))
print(f"✓ Calculated {valid_psat}/{N_POINTS} points along Psat curve")

## Step 7: Plot Low Pressure Results (<1 kbar)

Compare calculated solubilities with experimental data at low pressures.

In [ ]:
print("\nCreating plots...")

if len(exp_data) > 0:
    # Separate low and high pressure data
    low_P_threshold = 1.0
    non_psat_data = exp_data[~exp_data["is_psat"]]
    psat_data = exp_data[exp_data["is_psat"]]

    low_P_data = non_psat_data[non_psat_data["P_kbar"] < low_P_threshold]
    low_P_pressures = sorted(low_P_data["P_kbar"].unique()) if len(low_P_data) > 0 else []

    # Author markers
    author_markers = {
        "Kennedy_1950": "o",
        "Hemley_1980": "^",
        "Morey_Fournier_Rowe_1962": "s",
        "Walther_Orville_1983": "D",
        "Manning_1994": "v",
        "Newton_Manning_2000": "p",
    }

    # Plot 1: Low Pressure
    fig1, ax1 = plt.subplots(figsize=(14, 8))

    # Colors for low pressures
    n_low = max(len(low_P_pressures), 1)
    colors_low = plt.cm.viridis(np.linspace(0, 0.9, n_low))
    P_to_color_low = {P: colors_low[i % len(colors_low)] for i, P in enumerate(low_P_pressures)}

    # Plot experimental data
    for P_kbar in low_P_pressures:
        P_tol = 0.05 * P_kbar if P_kbar > 0.1 else 0.01
        subset = low_P_data[
            (low_P_data["P_kbar"] >= P_kbar - P_tol) &
            (low_P_data["P_kbar"] <= P_kbar + P_tol)
        ]
        if len(subset) == 0:
            continue

        for author in subset["reference"].unique():
            author_subset = subset[subset["reference"] == author]
            marker = author_markers.get(author, "o")
            ax1.scatter(
                author_subset["T_C"],
                author_subset["molality_m"],
                c=[P_to_color_low[P_kbar]],
                marker=marker,
                s=70,
                alpha=0.7,
                edgecolors="black",
                linewidths=0.4,
                label=f"Exp P={P_kbar:.2f} kbar ({author})",
                zorder=10
            )

    # Plot Psat experimental data
    if len(psat_data) > 0:
        low_psat_data = psat_data[(psat_data["P_kbar"] < 1.0) | (psat_data["P_kbar"].isna())]
        for author in low_psat_data["reference"].unique():
            author_psat = low_psat_data[low_psat_data["reference"] == author]
            marker = author_markers.get(author, "s")
            ax1.scatter(
                author_psat["T_C"],
                author_psat["molality_m"],
                c="purple",
                marker=marker,
                s=80,
                alpha=0.8,
                edgecolors="darkviolet",
                linewidths=0.5,
                label=f"Exp P=Psat ({author})",
                zorder=11
            )

    # Plot calculated curves
    for P_kbar in low_P_pressures:
        if P_kbar not in solubility_curves:
            continue
        curve = solubility_curves[P_kbar]
        valid = ~np.isnan(curve["molality"])
        ax1.plot(
            curve["T_C"][valid],
            curve["molality"][valid],
            color=P_to_color_low[P_kbar],
            linewidth=2.0,
            linestyle="-",
            label=f"Calc P={P_kbar:.2f} kbar",
            zorder=5
        )

    # Plot Psat curve
    if "Psat" in solubility_curves:
        curve_psat = solubility_curves["Psat"]
        valid_psat_m = ~np.isnan(curve_psat["molality"])
        ax1.plot(
            curve_psat["T_C"][valid_psat_m],
            curve_psat["molality"][valid_psat_m],
            color="purple",
            linewidth=3.0,
            linestyle="-",
            label="Calc P=Psat",
            zorder=6,
            alpha=0.9
        )

    ax1.set_yscale("log")
    ax1.set_ylim(1e-4, 1e-1)
    ax1.set_xlabel("Temperature (°C)", fontsize=14, fontweight="bold")
    ax1.set_ylabel("Quartz Solubility (mol/kg-H₂O)", fontsize=14, fontweight="bold")
    ax1.set_title("Quartz Solubility: Low Pressure (<1 kbar)", fontsize=16, fontweight="bold", pad=20)
    ax1.grid(True, which="both", alpha=0.3, linestyle="--")

    info_text = "DEW24 (species) + SUPCRTBL (quartz) + Zhang-Duan 2005 EOS (H₂O)"
    ax1.text(
        0.02, 0.98, info_text,
        transform=ax1.transAxes,
        fontsize=8,
        verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5)
    )

    ax1.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9, framealpha=0.9, ncol=1)
    plt.tight_layout()
    plt.savefig("quartz_solubility_comparison_low_P_dew24.png", dpi=300, bbox_inches="tight")
    print("✓ Figure 1 saved: quartz_solubility_comparison_low_P_dew24.png")
    plt.show()
else:
    print("⚠ No experimental data available for plotting")

## Step 8: Plot High Pressure Results (≥1 kbar)

Compare calculated solubilities with experimental data at high pressures.

In [ ]:
if len(exp_data) > 0:
    # High pressure data
    high_P_data = non_psat_data[non_psat_data["P_kbar"] >= low_P_threshold]

    # Collect all high pressure experiments (including psat >= 1 kbar)
    high_P_all_data = pd.concat([
        psat_data[(psat_data["P_kbar"] >= 1.0) & (psat_data["P_kbar"].notna())],
        non_psat_data[non_psat_data["P_kbar"] >= 1.0]
    ], ignore_index=True)

    high_P_all_pressures = sorted(high_P_all_data["P_kbar"].unique())

    # Plot 2: High Pressure
    fig2, ax2 = plt.subplots(figsize=(14, 8))

    # Colors for high pressures
    n_high = max(len(high_P_all_pressures), 1)
    colors_high = plt.cm.plasma(np.linspace(0, 0.9, n_high))
    P_to_color_high = {P: colors_high[i % len(colors_high)] for i, P in enumerate(high_P_all_pressures)}

    # Plot experimental data
    for P_kbar in high_P_all_pressures:
        P_tol = 0.05 * P_kbar
        subset = high_P_all_data[
            (high_P_all_data["P_kbar"] >= P_kbar - P_tol) &
            (high_P_all_data["P_kbar"] <= P_kbar + P_tol)
        ]
        if len(subset) == 0:
            continue

        for author in subset["reference"].unique():
            author_subset = subset[subset["reference"] == author]
            marker = author_markers.get(author, "o")
            ax2.scatter(
                author_subset["T_C"],
                author_subset["molality_m"],
                c=[P_to_color_high[P_kbar]],
                marker=marker,
                s=70,
                alpha=0.7,
                edgecolors="black",
                linewidths=0.4,
                label=f"Exp P={P_kbar:.2f} kbar ({author})",
                zorder=10
            )

    # Plot calculated curves
    for P_kbar in high_P_all_pressures:
        if P_kbar not in solubility_curves:
            continue
        curve = solubility_curves[P_kbar]
        valid = ~np.isnan(curve["molality"])
        ax2.plot(
            curve["T_C"][valid],
            curve["molality"][valid],
            color=P_to_color_high[P_kbar],
            linewidth=2.0,
            linestyle="-",
            label=f"Calc P={P_kbar:.2f} kbar",
            zorder=5
        )

    ax2.set_yscale("log")
    ax2.set_xlabel("Temperature (°C)", fontsize=14, fontweight="bold")
    ax2.set_ylabel("Quartz Solubility (mol/kg-H₂O)", fontsize=14, fontweight="bold")
    ax2.set_title("Quartz Solubility: High Pressure (>=1 kbar)", fontsize=16, fontweight="bold", pad=20)
    ax2.grid(True, which="both", alpha=0.3, linestyle="--")

    info_text = "DEW24 (species) + SUPCRTBL (quartz) + Zhang-Duan 2005 EOS (H₂O)"
    ax2.text(
        0.02, 0.98, info_text,
        transform=ax2.transAxes,
        fontsize=8,
        verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5)
    )

    ax2.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=9, framealpha=0.9, ncol=1)
    plt.tight_layout()
    plt.savefig("quartz_solubility_comparison_high_P_dew24.png", dpi=300, bbox_inches="tight")
    print("✓ Figure 2 saved: quartz_solubility_comparison_high_P_dew24.png")
    plt.show()

## Step 9: Plot Residuals (Measured - Predicted)

Analyze the differences between experimental measurements and model predictions.

In [ ]:
if len(exp_data) > 0:
    print("\nCalculating residuals...")

    # Predict molality at each experimental (T, P)
    solver_resid = EquilibriumSolver(system)
    predicted = []

    for _, row in exp_data.iterrows():
        T_C = float(row["T_C"])
        P_kbar_row = row["P_kbar"]

        # For NaN pressures (on Psat), compute Psat
        if pd.isna(P_kbar_row):
            P_kbar_row = psat_kbar(T_C)

        if pd.isna(P_kbar_row):
            predicted.append(np.nan)
            continue

        P_bar = float(P_kbar_row) * 1000.0

        state = ChemicalState(system)
        state.set("WATER,AQ", 1.0, "kg")
        state.set("H+", 1e-8, "mol")
        state.set("OH-", 1e-8, "mol")
        state.set("SiO2_aq", 1e-6, "mol")
        state.set("Quartz", 10.0, "mol")
        state.pressure(P_bar, "bar")
        state.temperature(float(T_C), "celsius")

        result = solver_resid.solve(state)
        if result.succeeded():
            try:
                aqprops = AqueousProps(state)
                molality = float(aqprops.speciesMolality("SiO2_aq"))
            except:
                molality = np.nan
        else:
            molality = np.nan

        predicted.append(molality)

    exp_resid = exp_data.copy()
    exp_resid["predicted_m"] = predicted
    exp_resid["abs_diff"] = exp_resid["molality_m"] - exp_resid["predicted_m"]
    exp_resid["rel_diff_pct"] = np.where(
        exp_resid["predicted_m"] > 0,
        100.0 * exp_resid["abs_diff"] / exp_resid["predicted_m"],
        np.nan
    )

    # Plot residuals
    finite_pressures = exp_resid["P_kbar"].dropna()
    if len(finite_pressures) > 0:
        norm = plt.Normalize(vmin=finite_pressures.min(), vmax=finite_pressures.max())
    else:
        norm = plt.Normalize(vmin=0.0, vmax=1.0)
    cmap = plt.cm.plasma

    fig_res, (ax_abs, ax_rel) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

    for _, row in exp_resid.iterrows():
        T_C = row["T_C"]
        P_kbar_row = row["P_kbar"]
        author = row["reference"]
        marker = author_markers.get(author, "o")

        if pd.isna(P_kbar_row):
            color = "purple"
        else:
            color = cmap(norm(P_kbar_row))

        ax_abs.scatter(
            T_C, row["abs_diff"],
            c=[color], marker=marker, s=50,
            alpha=0.7, edgecolors="black", linewidths=0.4
        )

        ax_rel.scatter(
            T_C, row["rel_diff_pct"],
            c=[color], marker=marker, s=50,
            alpha=0.7, edgecolors="black", linewidths=0.4
        )

    ax_abs.axhline(0, color="gray", linestyle="--", linewidth=1.0)
    ax_abs.set_ylabel("Measured - Predicted (mol/kg)", fontsize=12, fontweight="bold")
    ax_abs.grid(True, alpha=0.3, linestyle="--")

    ax_rel.axhline(0, color="gray", linestyle="--", linewidth=1.0)
    ax_rel.set_ylabel("Relative Diff (%)", fontsize=12, fontweight="bold")
    ax_rel.set_xlabel("Temperature (°C)", fontsize=12, fontweight="bold")
    ax_rel.grid(True, alpha=0.3, linestyle="--")

    plt.tight_layout()
    plt.savefig("quartz_solubility_residuals_dew24.png", dpi=300, bbox_inches="tight")
    print("✓ Figure 3 saved: quartz_solubility_residuals_dew24.png")
    plt.show()

print("\n" + "=" * 70)
print("✓ Analysis complete!")
print("=" * 70)